# CoStar Whole - Geographic Identifier Extraction

**Goal:** Extract geographic identifiers from the full CoStar dataset for geocoding.
**Scope:** Whole dataset (all markets)
**Source:** `data/0_raw/CoStar_Whole.csv`
**Output:** Chunked files `1_whole_properties_for_geocoding_001.csv`, etc. (≤100MB each for GitHub)

In [1]:
import os
import pandas as pd

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 100)

In [2]:
# --- Config: whole dataset, chunked output for GitHub 100MB limit ---
DATA_LABEL = "whole"
DATA_PATH = os.path.join("..", "..", "data", "0_raw", "CoStar_Whole.csv")
OUT_DIR = os.path.join("..", "..", "data", "1_derived")
MAX_ROWS_PER_FILE = 100_000
READ_CHUNKSIZE = 10_000

GEO_COLS = [
    "LeaseCompId", "PropertyId", "PropertyName", "Tenant.Name", "TenantIndustry", "TenantNAICS",
    "Market", "Submarket", "Address.DeliveryAddress", "Suite", "Address.Locality",
    "Address.County", "Address.PostalCode", "Address.RegionName", "Address.Subdivision",
    "Address.Country", "IsVerified.RawValue", "SpaceUse", "LeaseSource", "LeaseOrigin",
]

def build_full_address(row):
    addr = str(row.get("Address.DeliveryAddress", "") or "").strip()
    loc = str(row.get("Address.Locality", "") or "").strip()
    sub = str(row.get("Address.Subdivision", "") or "").strip()
    zipv = row.get("Address.PostalCode", "")
    zipv = "" if pd.isna(zipv) else str(int(zipv)) if isinstance(zipv, float) else str(zipv)
    country = str(row.get("Address.Country", "") or "").strip()
    parts = [p for p in [addr, loc, (sub + " " + zipv).strip(), country] if p]
    return ", ".join(parts).replace(", ,", ",").strip(", ")

seen_property_ids = set()
prop_buffer = []
file_counter = 0
total_rows_read = 0
total_properties_written = 0
os.makedirs(OUT_DIR, exist_ok=True)

for chunk in pd.read_csv(DATA_PATH, encoding="latin1", on_bad_lines="skip", chunksize=READ_CHUNKSIZE):
    total_rows_read += len(chunk)
    present = [c for c in GEO_COLS if c in chunk.columns]
    if not present:
        continue
    geo = chunk[present].copy()
    geo["full_address"] = geo.apply(build_full_address, axis=1)
    for _, row in geo.iterrows():
        pid = row.get("PropertyId")
        if pd.isna(pid):
            continue
        pid = int(pid) if isinstance(pid, float) else pid
        if pid in seen_property_ids:
            continue
        seen_property_ids.add(pid)
        prop_buffer.append(row)
        if len(prop_buffer) >= MAX_ROWS_PER_FILE:
            file_counter += 1
            out_name = f"1_{DATA_LABEL}_properties_for_geocoding_{file_counter:03d}.csv"
            out_path = os.path.join(OUT_DIR, out_name)
            pd.DataFrame(prop_buffer).to_csv(out_path, index=False)
            total_properties_written += len(prop_buffer)
            print(f"Checkpoint: saved {len(prop_buffer):,} rows → {out_name} (total: {total_properties_written:,})")
            prop_buffer = []

# Write final chunk
if prop_buffer:
    file_counter += 1
    out_name = f"1_{DATA_LABEL}_properties_for_geocoding_{file_counter:03d}.csv"
    out_path = os.path.join(OUT_DIR, out_name)
    pd.DataFrame(prop_buffer).to_csv(out_path, index=False)
    total_properties_written += len(prop_buffer)
    print(f"Final: saved {len(prop_buffer):,} rows → {out_name}")

print(f"\nDone. Read {total_rows_read:,} rows → {total_properties_written:,} unique properties in {file_counter} file(s)")
sample_path = os.path.join(OUT_DIR, f"1_{DATA_LABEL}_properties_for_geocoding_001.csv")
df = pd.read_csv(sample_path, nrows=5) if os.path.exists(sample_path) else pd.DataFrame()
df.head(3)

C:\Users\clint\AppData\Local\Temp\ipykernel_10556\2970671959.py:32: DtypeWarning: Columns (216,217,219,220,221,223) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(DATA_PATH, encoding="latin1", on_bad_lines="skip", chunksize=READ_CHUNKSIZE):
C:\Users\clint\AppData\Local\Temp\ipykernel_10556\2970671959.py:32: DtypeWarning: Columns (4,190) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(DATA_PATH, encoding="latin1", on_bad_lines="skip", chunksize=READ_CHUNKSIZE):
C:\Users\clint\AppData\Local\Temp\ipykernel_10556\2970671959.py:32: DtypeWarning: Columns (216,217,219,220,221,223,224,226) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(DATA_PATH, encoding="latin1", on_bad_lines="skip", chunksize=READ_CHUNKSIZE):
C:\Users\clint\AppData\Local\Temp\ipykernel_10556\2970671959.py:32: DtypeWarning: Columns (216,217,219,220,221,223) have mixe

Checkpoint: saved 100,000 rows → 1_whole_properties_for_geocoding_001.csv (total: 100,000)


C:\Users\clint\AppData\Local\Temp\ipykernel_10556\2970671959.py:32: DtypeWarning: Columns (216,217,219,220,221,223) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(DATA_PATH, encoding="latin1", on_bad_lines="skip", chunksize=READ_CHUNKSIZE):
C:\Users\clint\AppData\Local\Temp\ipykernel_10556\2970671959.py:32: DtypeWarning: Columns (216,217,219,220,221,223) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(DATA_PATH, encoding="latin1", on_bad_lines="skip", chunksize=READ_CHUNKSIZE):
C:\Users\clint\AppData\Local\Temp\ipykernel_10556\2970671959.py:32: DtypeWarning: Columns (4,143,190,216,217,219,220,221,223) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(DATA_PATH, encoding="latin1", on_bad_lines="skip", chunksize=READ_CHUNKSIZE):
C:\Users\clint\AppData\Local\Temp\ipykernel_10556\2970671959.py:32: DtypeWarning: Columns (216,217,219,22

Checkpoint: saved 100,000 rows → 1_whole_properties_for_geocoding_002.csv (total: 200,000)


C:\Users\clint\AppData\Local\Temp\ipykernel_10556\2970671959.py:32: DtypeWarning: Columns (4,143,190,216,217,219,220,221,223) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(DATA_PATH, encoding="latin1", on_bad_lines="skip", chunksize=READ_CHUNKSIZE):
C:\Users\clint\AppData\Local\Temp\ipykernel_10556\2970671959.py:32: DtypeWarning: Columns (143) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(DATA_PATH, encoding="latin1", on_bad_lines="skip", chunksize=READ_CHUNKSIZE):
C:\Users\clint\AppData\Local\Temp\ipykernel_10556\2970671959.py:32: DtypeWarning: Columns (4,190) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(DATA_PATH, encoding="latin1", on_bad_lines="skip", chunksize=READ_CHUNKSIZE):
C:\Users\clint\AppData\Local\Temp\ipykernel_10556\2970671959.py:32: DtypeWarning: Columns (216,217,219,220,221,223) have mixed types. Specify d

...

C:\Users\clint\AppData\Local\Temp\ipykernel_10556\2970671959.py:32: DtypeWarning: Columns (4,190,216,217,219,220,221,223) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(DATA_PATH, encoding="latin1", on_bad_lines="skip", chunksize=READ_CHUNKSIZE):
C:\Users\clint\AppData\Local\Temp\ipykernel_10556\2970671959.py:32: DtypeWarning: Columns (216,217,219,220,221,223) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(DATA_PATH, encoding="latin1", on_bad_lines="skip", chunksize=READ_CHUNKSIZE):
C:\Users\clint\AppData\Local\Temp\ipykernel_10556\2970671959.py:32: DtypeWarning: Columns (216,217,219,220,221,223) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(DATA_PATH, encoding="latin1", on_bad_lines="skip", chunksize=READ_CHUNKSIZE):
C:\Users\clint\AppData\Local\Temp\ipykernel_10556\2970671959.py:32: DtypeWarning: Columns (143) have mixed ty

Checkpoint: saved 100,000 rows → 1_whole_properties_for_geocoding_010.csv (total: 1,000,000)


C:\Users\clint\AppData\Local\Temp\ipykernel_10556\2970671959.py:32: DtypeWarning: Columns (4,143,190,216,217,219,220,221,223) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(DATA_PATH, encoding="latin1", on_bad_lines="skip", chunksize=READ_CHUNKSIZE):
C:\Users\clint\AppData\Local\Temp\ipykernel_10556\2970671959.py:32: DtypeWarning: Columns (216,217,219,220,221,223) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(DATA_PATH, encoding="latin1", on_bad_lines="skip", chunksize=READ_CHUNKSIZE):
C:\Users\clint\AppData\Local\Temp\ipykernel_10556\2970671959.py:32: DtypeWarning: Columns (143) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(DATA_PATH, encoding="latin1", on_bad_lines="skip", chunksize=READ_CHUNKSIZE):
C:\Users\clint\AppData\Local\Temp\ipykernel_10556\2970671959.py:32: DtypeWarning: Columns (143,216,217,219,220,221,223,224,22

Final: saved 51,219 rows → 1_whole_properties_for_geocoding_011.csv

Done. Read 5,272,455 rows → 1,051,219 unique properties in 11 file(s)


,LeaseCompId,PropertyId,PropertyName,Tenant.Name,TenantIndustry,TenantNAICS,Market,Submarket,Address.DeliveryAddress,Suite,...,Address.County,Address.PostalCode,Address.RegionName,Address.Subdivision,Address.Country,IsVerified.RawValue,SpaceUse,LeaseSource,LeaseOrigin,full_address
0,114028931.0,435558.0,Bldg A,Robert A Garrett,Construction,NaN,Atlanta,Cumberland/Galleria,4501 Circle 75 Pky SE,1170.0,...,Cobb,30339.0,NaN,GA,United States,False,Office,Third Party - Landlord Rep,CoStar Research,"4501 Circle 75 Pky SE, Atlanta, GA 30339, United States"
1,113952192.0,445018.0,The Palisades Medical Office,Physio,Health Care and Social Assistance,"Offices of Physical, Occupational and Speech Therapists, and Audiologists",Atlanta,Upper Buckhead,3200 Downwood Cir NW,350.0,...,Fulton,30327.0,NaN,GA,United States,False,Office,Third Party - Landlord Rep,CoStar Research,"3200 Downwood Cir NW, Atlanta, GA 30327, United States"
2,114008333.0,440921.0,100 Edgewood,Georgia Council on Substance Abuse,Health Care and Social Assistance,Individual and Family Services,Atlanta,Downtown Atlanta,100 Edgewood Ave NE,540.0,...,Fulton,30303.0,NaN,GA,United States,False,Office,Third Party - Landlord Rep,CoStar Research,"100 Edgewood Ave NE, Atlanta, GA 30303, United States"
